### Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import scipy.stats as stats
from pyprojroot.here import here
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from sklearn.metrics import log_loss
from sklearn.metrics import accuracy_score
import pickle
from pathlib import Path

In [2]:
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "processed_output" / "X_processed.csv").exists())
DATA_DIR = ROOT / "processed_output"
MODEL_DIR = ROOT / "models"

### Safety Checks on Datasets

In [3]:
X_processed = pd.read_csv(DATA_DIR / "X_processed.csv", index_col=0)
y_frame = pd.read_csv(DATA_DIR / "y_processed.csv", index_col=0)
if y_frame.shape[1] != 1 or not X_processed.index.equals(y_frame.index):
    raise ValueError("Expected one target column aligned with the feature row index.")
if not X_processed.index.is_unique or not X_processed.columns.is_unique:
    raise ValueError("Feature row indices and column names must be unique.")
y_processed = y_frame.iloc[:, 0]
if set(y_processed.unique()) != {0, 1}:
    raise ValueError("Expected a binary 0/1 target without missing values.")
if not all(pd.api.types.is_numeric_dtype(t) for t in X_processed.dtypes):
    raise ValueError("Expected numeric processed features.")
if not np.isfinite(X_processed.to_numpy()).all():
    raise ValueError("Features must not contain missing or infinite values.")

In [4]:
# Baseline B retains all processed features, including demographics.
excluded_columns = []
print(f"Baseline B uses all {X_processed.shape[1]} features.")

Baseline B uses all 51 features.


### Dataset Statistics and Splitting

In [5]:
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y_processed, test_size=0.2, random_state=42, stratify=y_processed
)
print(f"Training: {X_train.shape[0]} samples, {X_train.shape[1]} features")
print(f"Test: {X_test.shape[0]} samples, {X_test.shape[1]} features")
print("Training class counts:", y_train.value_counts().to_dict())

Training: 351909 samples, 51 features
Test: 87978 samples, 51 features
Training class counts: {0: 337839, 1: 14070}


### Random Forest Modelling Structure

In [ ]:
def train_test_RFC(X_train, X_test, y_train, y_test, estimators, min_sample_split, max_features, bootstrap_bool, weight, min_sample_leaf = 20, maxdepth = 10, njobs = -1):
    # Create a random forest model
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import StandardScaler
    model = make_pipeline(RandomForestClassifier(
    n_estimators = estimators,       # Number of trees in the forest
    max_depth= maxdepth,           # Restrict depth to prevent overfitting
    min_samples_split = min_sample_split,
    min_samples_leaf = min_sample_leaf, # Smoothing the leaves helps reduce variance in noisy data.
    max_features = max_features,
    bootstrap= bootstrap_bool,            # Use bootstrapping to sample data for each tree
    random_state= 42,        # Ensures reproducible results across production runs
    n_jobs= njobs,             # Use all available CPU cores for parallel processing
    class_weight= weight
))

    # Fit the model to the training data
    model.fit(X_train, y_train)

    #Predict the probability of the class of examples of the test data

    proba_predict = model.predict_proba(X_test)

    #Computing the log loss of the probability (proba_predict) by comparing 
    #the actual labels of the test data to the probability predictions

    logloss = log_loss(y_test, proba_predict)

    #Making predictions on the test data, to evaluate the accuracy score
    
    class_label_predict = model.predict(X_test)

    #Computing the accuracy score of the model by comparing the y_test
    #with the class label predictions generated from the step above
    
    accscore = accuracy_score(y_test, class_label_predict)
    
    return {"model": model, "predicted_proba": proba_predict, 'accuracy_score': accscore, 'log_loss': logloss, "predicted_classes": class_label_predict}

### Model Pickling

In [6]:
def pickling(model, model_name):
    with open(model_name, "wb") as handle:
        pickle.dump(model, handle, protocol=pickle.HIGHEST_PROTOCOL)
    # Only load pickle files produced here or from another trusted source.
    with open(model_name, "rb") as handle:
        return pickle.load(handle)

## Summary



This notebook models a secondary baseline Random Forest classifier using all available features to allow for performance comparisons. Key steps include:   


Data Ingestion & Validation: Executes the same data loading and strict integrity checks as the first notebook.   

Feature Selection: Retains all 51 features, explicitly keeping the gender, ethnicity, and region columns to build Baseline B.   

Data Splitting: Stratifies the data into training and test sets maintaining the exact same sample sizes (351,909 train / 87,978 test) but utilizing the full 51-feature dataset.   

Model Pipeline & Persistence: Utilizes the same train_test_RFC pipeline and pickling functions to train, evaluate, and save the inclusive classifier.   
